In [1]:
# import necessary dependencies and load in dataset
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

df = pd.read_pickle("FallAllD.pkl")
print(df.head())

   SubjectID  Device  ActivityID  TrialNo  \
0          1       1          13        1   
1          1       1          13        2   
2          1       1          13        3   
3          1       1          13        4   
4          1       1          13        5   

                                                 Acc  \
0  [[4155, 55, -82], [4157, 55, -126], [4157, 50,...   
1  [[4002, -50, 156], [4002, -38, 172], [4003, -2...   
2  [[3983, 40, -335], [3984, 38, -324], [3987, 34...   
3  [[3959, 165, 197], [3953, 165, 196], [3951, 16...   
4  [[3750, -182, 252], [3767, -166, 259], [3781, ...   

                                                 Bar  \
0  [[1013.193859563175, 23.71569457530976], [1013...   
1  [[1013.214612231942, 24.11322125434876], [1013...   
2  [[1013.25438772214, 24.49373892784119], [1013....   
3  [[1013.167161107663, 24.72971366882324], [1013...   
4  [[1013.304382379492, 25.49256420612335], [1013...   

                                                 Gyr  \

In [2]:
# map of activity labels form ActivityID2Str.m
ACTIVITY_LABELS = {
    1:'Start clap hands',2:'Clap hands',3:'Stop clap hands',4:'Clap hands 1',
    5:'Start wave hands',6:'Wave hands',7:'Stop wave hands',8:'Raising hand up',
    9:'Moving hand down',10:'Move hand up -> down',11:'Hand shaking',12:'Beating a table',
    13:'Sitting down',14:'Standing up',15:'Fail to stand up',16:'Lying down',
    17:'Turning while lying',18:'Rising up',19:'Start walking',20:'Walking slowly',
    21:'Stop walking',22:'Walking quickly',23:'Stumbling',24:'Jogging slowly',
    25:'Jogging quickly',26:'Jumping slightly',27:'Jumping strongly',28:'Bending down',
    29:'Start going upstairs',30:'Going upstairs',31:'Stop going upstairs',
    32:'Start going downstairs',33:'Going downstairs',34:'Stop going downstairs',
    35:'Upstairs quickly',36:'Downstairs quickly',37:'Start ascending, lift',
    38:'Stop ascending, lift',39:'Start descending, lift',40:'Stop descending, lift',
    41:'Standing, moving bus',42:'Sitting, moving bus',43:'Start jogging',44:'Stop jogging',
    101:'Fall F, walking, trip',102:'Fall F, walking, trip, rec.',103:'Fall F, walking, slip',
    104:'Fall F, walking, slip, rec.',105:'Fall F, walking, slip, rot.',
    106:'Fall F, walking, slip, rot., rec.',107:'Fall B, walking, slip',
    108:'Fall B, walking, slip, rec.',109:'Fall B, walking, slip, rot.',
    110:'Fall B, walking, slip rot., rec.',111:'Fall F, walking, syncope',
    112:'Fall B, walking, syncope',113:'Fall L, walking, syncope',114:'Fall, syncope, table',
    115:'Fall F, try sit',116:'Fall F, try sit, rec.',117:'Fall B, try sit',
    118:'Fall B, try sit, rec.',119:'Fall L, try sit',120:'Fall L, try sit, rec.',
    121:'Fall F, jog, trip',122:'Fall F, jog, trip, rec.',123:'Fall F, jog, slip',
    124:'Fall F, jog, slip, rev.',125:'Fall F, jog, slip, rot.',
    126:'Fall F, jog, slip, rot., rec.',127:'Fall L, bed',128:'Fall L, bed, rec.',
    129:'Fall F, chair, syncope',130:'Fall B, chair, syncope',131:'Fall L, chair, syncope',
    132:'Fall F, syncope',133:'Fall B, syncope',134:'Fall L, syncope',
    135:'Fall, syncope, slide over a wall',
}

df['ActivityLabel'] = df['ActivityID'].map(ACTIVITY_LABELS)

In [3]:
# Check for missing values in each column and cell (entire Arrays NAN and values within each array as NAN)
print(df.isnull().sum()) 

def count_inner_nans(col):
    return df[col].apply(lambda c: np.isnan(np.array(c, dtype=float)).sum()).sum()

for col in ['Acc','Bar','Gyr','Mag']:
    print(f"{col}: inner NaNs = {count_inner_nans(col)}")

SubjectID        0
Device           0
ActivityID       0
TrialNo          0
Acc              0
Bar              0
Gyr              0
Mag              0
ActivityLabel    0
dtype: int64
Acc: inner NaNs = 0
Bar: inner NaNs = 0
Gyr: inner NaNs = 0
Mag: inner NaNs = 0


In [4]:
# Check for duplicate trials (same subject/device/activity/trial number)
dupe_count = df.duplicated(subset=['SubjectID','Device','ActivityID','TrialNo']).sum()
print(f"Duplicate trials: {dupe_count}")

Duplicate trials: 0


In [5]:
# Detect and treat outliers
# determine the typical magnitude ranges for each of the sensors and print out the min/max/mean/std for each 
def array_stats(col):
    all_vals = np.vstack(df[col].values)
    print(f"{col}: min={all_vals.min(axis=0)}, max={all_vals.max(axis=0)}, "
          f"mean={all_vals.mean(axis=0)}, std={all_vals.std(axis=0)}")

for col in ['Acc','Bar','Gyr','Mag']:
    array_stats(col)
    

Acc: min=[-32764 -32764 -32764], max=[32764 32764 32764], mean=[2629.01307747 -109.1029872  -411.31289003], std=[2319.52175304 1924.14661666 1829.00754101]
Bar: min=[989.34982751  16.79121413], max=[1029.71466082   37.69050867], mean=[1012.71611694   27.78424432], std=[8.52112461 2.39804653]
Gyr: min=[-30568 -32768 -27849], max=[32715 32699 24621], mean=[-8.1240047  40.34886539 18.65250654], std=[776.91045904 579.29349239 631.56605799]
Mag: min=[-7267 -6350 -9178], max=[10085  9915 22100], mean=[2158.13881236  593.65318036 3526.99152072], std=[1737.35034631 1658.74920982 2820.98426434]


In [6]:
# use the valid ranges based on the paper stated in section Sensors Configurations and
# in SECTION III: The Proposed Data Acquisition System
VALID_RANGES = {
    'Acc': (-32768, 32767),   # ±8 g configured full-scale (LSM9DS1)
    'Gyr': (-28571, 28571),   # ±2000 dps configured full-scale (LSM9DS1)
    'Mag': (-28571, 28571),   # ±4 Gauss configured full-scale (LSM9DS1)
    'Bar': (10, 1200),        # mbar/hPa, MS5607-02BA03 linear operating range
}

# Bounds are the sensor's hardware/physical limits
# That's because accelerometer/gyroscope have large peaks that are still within sensor range that show a fall and not an outlier
# so this function determine if one of the values is out of range (aka outlier) of what the hardware can physically do
def flag_invalid(cell, valid_range):
    arr = np.array(cell, dtype=float)
    low, high = valid_range
    # non finite or out of sensor range 
    return ~np.isfinite(arr) | (arr < low) | (arr > high)

for col in ['Acc','Bar','Gyr','Mag']:
    low, high = VALID_RANGES[col]
    df[f'{col}_invalid_count'] = df[col].apply(lambda count: flag_invalid(count, (low,high)).sum())

print(df[[f'{count}_invalid_count' for count in ['Acc','Bar','Gyr','Mag']]].sum())

Acc_invalid_count     0
Bar_invalid_count     0
Gyr_invalid_count    47
Mag_invalid_count     0
dtype: int64


In [7]:
# instead of removing the sensor values, replace the outliers (47 from gyr) with other valid values and interpolated estimates 
# from neighboring samples to get the same signal shape. 
def interpolate_invalid(cell, valid_range):
    arr = np.array(cell, dtype=float)
    low, high = valid_range
    mask = ~np.isfinite(arr) | (arr < low) | (arr > high)
    arr[mask] = np.nan
    arr = pd.DataFrame(arr).interpolate(method='linear', limit_direction='both').values
    return arr.tolist()

# look into which activity was the one that had the most out of range values (it was activity 39 which could indicate the 
# lift beginning to move). Only 24 samples from this trial were invalid so fine to interpolate and everything else was btw 1-5
affected = df[df['Gyr_invalid_count'] > 0][['SubjectID','Device','ActivityID','TrialNo','Gyr_invalid_count']]
print(affected)

# Only Gyr had 47 invalid samples and the rest (Acc, Bar, Mag) were all 0, so nothing needs to be done for those
df['Gyr'] = df['Gyr'].apply(lambda count: interpolate_invalid(count, VALID_RANGES['Gyr']))
df['Gyr_invalid_count_after'] = df['Gyr'].apply(lambda count: flag_invalid(count, VALID_RANGES['Gyr']).sum())
# should now be 0
print(df['Gyr_invalid_count_after'].sum())  

      SubjectID  Device  ActivityID  TrialNo  Gyr_invalid_count
1525          3       2         127        3                  1
1978          4       3          39        2                 24
2086          5       1          27        4                  5
2160          5       1         123        2                  1
2161          5       1         124        2                  3
2169          5       1         128        2                  3
2171          5       1         128        4                  3
2179          5       1         134        2                  1
2524          5       3         107        2                  2
3337          8       3          14        2                  2
5188         12       3         121        1                  1
5195         12       3         124        2                  1
0


In [8]:
# Encode categorical variables
def encode_target(activity_id):
    """
    Encode ActivityID into binary target:
    ADL (0): IDs 1-44
    Fall (1): IDs 101-135
    """
    if 1 <= activity_id <= 44:
        return 0  # ADL
    elif 101 <= activity_id <= 135:
        return 1  # fall
    else:
        raise ValueError(f'Unknown activity: {activity_id}')
    
# target variable: Activity (fall vs not)
df['target'] = df['ActivityID'].apply(encode_target)

# verify the encoding
print(df['target'].value_counts())
print(df.groupby('target')['ActivityID'].agg(['min', 'max', 'nunique']))

# map the device position to the device number and verify that there are no NANs and only 3 categories
DEVICE_MAP = {1: 'Neck', 2: 'Wrist', 3: 'Waist'}
df['DevicePosition'] = df['Device'].map(DEVICE_MAP)
print(df['DevicePosition'].value_counts(dropna=False))

# encode the device position to be ready for the models to interpret 
device_encoded = pd.get_dummies(df['DevicePosition'], prefix='Device')
df = pd.concat([df, device_encoded], axis=1)

# determine which subjects did not have all 3 sensors
devices_associated_with_subjects = df.groupby('SubjectID')['DevicePosition'].apply(lambda x: set(x.unique()))
print(devices_associated_with_subjects)
incomplete_subjects = devices_associated_with_subjects[devices_associated_with_subjects.apply(len) < 3]
print("\nSubjects with incomplete device coverage:")
print(incomplete_subjects)

0    4883
1    1722
Name: target, dtype: int64
        min  max  nunique
target                   
0         1   44       44
1       101  135       35
Wrist    2515
Neck     2292
Waist    1798
Name: DevicePosition, dtype: int64
SubjectID
1     {Waist, Neck, Wrist}
2     {Waist, Neck, Wrist}
3     {Waist, Neck, Wrist}
4     {Waist, Neck, Wrist}
5     {Waist, Neck, Wrist}
6                  {Wrist}
7            {Waist, Neck}
8            {Waist, Neck}
9     {Waist, Neck, Wrist}
10    {Waist, Neck, Wrist}
11    {Waist, Neck, Wrist}
12    {Waist, Neck, Wrist}
13    {Waist, Neck, Wrist}
14    {Waist, Neck, Wrist}
15    {Waist, Neck, Wrist}
Name: DevicePosition, dtype: object

Subjects with incomplete device coverage:
SubjectID
6          {Wrist}
7    {Waist, Neck}
8    {Waist, Neck}
Name: DevicePosition, dtype: object


In [9]:
# Training, Validation and Test data split 
# using validation for tuning the model and adjusting all the hyperparameters in the future 

# split the data and shuffle it so its random between which subjects are selected for each train/val/test
subjects = df['SubjectID'].unique()
random_num = np.random.default_rng(42)
random_num.shuffle(subjects)

train_subjects = set(subjects[:11])
validation_subjects = set(subjects[11:13])
test_subjects = set(subjects[13:])

train_df = df[df['SubjectID'].isin(train_subjects)].copy()
val_df   = df[df['SubjectID'].isin(validation_subjects)].copy()
test_df  = df[df['SubjectID'].isin(test_subjects)].copy()

# verify the rows/subjects for each of the test/train/val data splits 
print(f"Train: {len(train_df)} rows, {len(train_subjects)} subjects")
print(f"Val:   {len(val_df)} rows, {len(validation_subjects)} subjects")
print(f"Test:  {len(test_df)} rows, {len(test_subjects)} subjects")

# check how many ADL vs fall for target are in each data split roughly 75/25 for each which is good 
print(train_df['target'].value_counts(normalize=True))
print(val_df['target'].value_counts(normalize=True))
print(test_df['target'].value_counts(normalize=True))

Train: 4483 rows, 11 subjects
Val:   1087 rows, 2 subjects
Test:  1035 rows, 2 subjects
0    0.741022
1    0.258978
Name: target, dtype: float64
0    0.721251
1    0.278749
Name: target, dtype: float64
0    0.750725
1    0.249275
Name: target, dtype: float64


In [10]:
# Feature Scaling and Normalization which standardizes each sensor channel to mean=0, std=1.
# This is needed because the raw sensor ranges differ wildly (Acc/Mag: thousands, Bar: ~1000, Gyr: hundreds-thousands) and
# those would otherwise dominate the model based on magnitude alone, not signal importance. Since we want to do a couple regressions, 
# this is necessary to scale. 
def fit_scaler(train_series):
    # stack all training samples together to compute one mean/std per axis
    all_vals = np.vstack(train_series.values)
    return all_vals.mean(axis=0), all_vals.std(axis=0)

def apply_scaler(cell, mean, std):
    # apply the train-derived mean/std to any split (train, val, or test)
    arr = np.array(cell, dtype=float)
    return ((arr - mean) / std).tolist()

# store fitted params per sensor, in case needed later
scalers = {} 
for col in ['Acc', 'Bar', 'Gyr', 'Mag']:
    # fit only on train data so that it doesn't overfit with test/val data leaked
    mean, std = fit_scaler(train_df[col])
    scalers[col] = (mean, std)
    # then apply to all three splits (including test and validation)
    for split_df in [train_df, val_df, test_df]:
        split_df[col] = split_df[col].apply(lambda c: apply_scaler(c, mean, std))

In [11]:
# Feature Engineering
# Classical models (SVM/RF/XGBoost) need one row of numbers per trial, all the same length. But our raw sensor data is a list 
# of readings that's a different length for every trial. So instead of feeding in the raw list, we summarize each trial's signal
# using a few numbers (mean, max, std, etc.) that describe its shape. That turns every trial into one same-sized row the model 
# can actually use.

def extract_features(arr):
    # mean/median/min/max/std: basic shape/spread of the signal
    # rms: energy-like magnitude measure, common for accelerometer/gyro signals
    arr = np.array(arr, dtype=float)
    return {
        'mean': arr.mean(axis=0),
        'median': np.median(arr, axis=0),
        'std': arr.std(axis=0),
        'max': arr.max(axis=0),
        'min': arr.min(axis=0),
        'rms': np.sqrt((arr**2).mean(axis=0)),
    }

# Axis names per sensor, used to build clear column names
# (Acc/Gyr/Mag are 3-axis; Bar has 2 channels: pressure and temperature)
AXIS_NAMES = {'Acc': ['x','y','z'], 'Gyr': ['x','y','z'], 'Mag': ['x','y','z'], 'Bar': ['pressure','temp']}

def build_feature_row(row):
    # For one trial (one dataframe row): extract all 6 stats x all axes x all 4 sensors,
    # flattened into a single dict when then becomes one row of the final feature matrix.
    features = {}
    for col in ['Acc', 'Gyr', 'Mag', 'Bar']:
        stats = extract_features(row[col])
        axes = AXIS_NAMES[col]
        for stat_name, values in stats.items():
            for axis_name, val in zip(axes, values):
                features[f'{col}_{stat_name}_{axis_name}'] = val 

    # Carry over device one-hot columns (Device_Neck/Wrist/Waist) as features too, 
    # so the model can learn device-position-specific patterns
    for c in df.columns:
        if c.startswith('Device_'):
            features[c] = row[c]

    features['target'] = row['target']
    return features

def build_feature_matrix(split_df):
    # Apply build_feature_row to every trial in a split, then separate features (X) from label (y)
    rows = split_df.apply(build_feature_row, axis=1)
    feat_df = pd.DataFrame(list(rows))
    y = feat_df.pop('target')
    X = feat_df
    return X, y

# Build flat feature matrices for each split independently so there is no leakage, since each
# split's stats (mean/std/etc.) are computed only from that split's own trials
X_train, y_train = build_feature_matrix(train_df)
X_val,   y_val   = build_feature_matrix(val_df)
X_test,  y_test  = build_feature_matrix(test_df)

# rows = trials, columns = engineered features
print(X_train.shape, X_val.shape, X_test.shape)
X_train.head()

(4483, 69) (1087, 69) (1035, 69)


,Acc_mean_x,Acc_mean_y,Acc_mean_z,Acc_median_x,Acc_median_y,Acc_median_z,Acc_std_x,Acc_std_y,Acc_std_z,Acc_max_x,...,Bar_std_temp,Bar_max_pressure,Bar_max_temp,Bar_min_pressure,Bar_min_temp,Bar_rms_pressure,Bar_rms_temp,Device_Neck,Device_Waist,Device_Wrist
0,0.599296,0.018139,0.054651,0.614081,0.006850,0.119349,0.098669,0.131253,0.321005,1.094361,...,0.018116,-0.106822,-1.592116,-0.136742,-1.655403,0.122333,1.624385,1,0,0
1,0.600542,0.009089,0.170381,0.614081,0.061433,0.190101,0.057592,0.093501,0.309189,0.840474,...,0.015520,-0.114609,-1.431044,-0.148415,-1.484426,0.129929,1.458397,1,0,0
2,0.611229,-0.078896,0.158565,0.615799,-0.073574,0.249723,0.109122,0.097515,0.263694,1.260612,...,0.014013,-0.115244,-1.273448,-0.170188,-1.320573,0.143106,1.298182,1,0,0
3,0.601989,0.029510,0.182178,0.612792,0.006059,0.244689,0.071922,0.054264,0.284340,0.983957,...,0.013009,-0.125844,-1.171927,-0.158261,-1.218531,0.141391,1.196582,1,0,0
4,0.588033,-0.302984,0.429627,0.590024,-0.326712,0.474434,0.102147,0.111411,0.275653,1.162236,...,0.019095,-0.109201,-0.825077,-0.151882,-0.890334,0.128325,0.858297,1,0,0


In [ ]:
# Feature Selection (fit on train only, apply unchanged to val/test)
# Since we generated a lot of stats per sensor, and some are useless (bc they barely change between trials) or redundant 
# (basically duplicate another feature), removing these makes the model simpler, faster, and less likely to overfit
# without losing any real information.

# Drop constant and near-constant features
selector = VarianceThreshold(threshold=0.01)
selector.fit(X_train)

kept_cols = X_train.columns[selector.get_support()]
dropped_cols = X_train.columns[~selector.get_support()]
print(f"Dropped {len(dropped_cols)} near-constant features: {list(dropped_cols)}")

X_train = X_train[kept_cols]
X_val   = X_val[kept_cols]
X_test  = X_test[kept_cols]

# Correlation filtering: so drop one of each highly correlated pair
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
print(f"Dropping {len(to_drop)} highly correlated features: {to_drop}")

X_train = X_train.drop(columns=to_drop)
X_val   = X_val.drop(columns=to_drop)
X_test  = X_test.drop(columns=to_drop)

print(f"Final feature count: {X_train.shape[1]}")

# verify data one last time and missing values should be 0
print(X_train.isnull().sum().sum())
print(X_train.shape, X_val.shape, X_test.shape)
# verify the final 54 feature names 
print(X_train.columns.tolist())

Dropped 0 near-constant features: []
Dropping 0 highly correlated features: []
Final feature count: 54
0
(4483, 54) (1087, 54) (1035, 54)
['Acc_mean_x', 'Acc_mean_y', 'Acc_mean_z', 'Acc_median_x', 'Acc_median_y', 'Acc_median_z', 'Acc_std_x', 'Acc_std_y', 'Acc_std_z', 'Acc_max_x', 'Acc_max_y', 'Acc_max_z', 'Acc_min_x', 'Acc_min_y', 'Acc_min_z', 'Acc_rms_x', 'Acc_rms_y', 'Acc_rms_z', 'Gyr_mean_x', 'Gyr_mean_y', 'Gyr_median_x', 'Gyr_median_y', 'Gyr_std_x', 'Gyr_std_y', 'Gyr_std_z', 'Gyr_max_x', 'Gyr_max_y', 'Gyr_max_z', 'Gyr_min_x', 'Gyr_min_y', 'Gyr_min_z', 'Mag_mean_x', 'Mag_mean_y', 'Mag_mean_z', 'Mag_median_x', 'Mag_std_x', 'Mag_std_y', 'Mag_std_z', 'Mag_max_x', 'Mag_max_y', 'Mag_max_z', 'Mag_min_x', 'Mag_min_y', 'Mag_min_z', 'Mag_rms_x', 'Mag_rms_y', 'Mag_rms_z', 'Bar_mean_pressure', 'Bar_mean_temp', 'Bar_rms_pressure', 'Bar_rms_temp', 'Device_Neck', 'Device_Waist', 'Device_Wrist']
